In [1]:
!pip uninstall protobuf mediapipe -y

Found existing installation: protobuf 3.20.3
Uninstalling protobuf-3.20.3:
  Successfully uninstalled protobuf-3.20.3


In [2]:
!pip install protobuf==3.20.3
!pip install mediapipe==0.8.6  # Replace with a version known to work with protobuf-3.20.3
!pip show protobuf mediapipe

  Using cached protobuf-3.20.3-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl.metadata (679 bytes)
Using cached protobuf-3.20.3-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl (1.1 MB)


ERROR: Could not find a version that satisfies the requirement mediapipe==0.8.6 (from versions: 0.9.1.0, 0.9.2.1, 0.9.3.0, 0.10.0, 0.10.1, 0.10.2, 0.10.3, 0.10.5, 0.10.7, 0.10.8, 0.10.9, 0.10.10, 0.10.11, 0.10.13, 0.10.14, 0.10.15)
ERROR: No matching distribution found for mediapipe==0.8.6
Name: protobuf
Version: 3.20.3
Summary: Protocol Buffers
Home-page: https://developers.google.com/protocol-buffers/
Author: 
Author-email: 
License: BSD-3-Clause
Location: /usr/local/lib/python3.10/dist-packages
Requires: 
Required-by: cudf-cu12, google-ai-generativelanguage, google-api-core, google-cloud-aiplatform, google-cloud-bigquery-connection, google-cloud-bigquery-storage, google-cloud-bigtable, google-cloud-datastore, google-cloud-firestore, google-cloud-functions, google-cloud-iam, google-cloud-language, google-cloud-pubsub, google-cloud-resource-manager, google-cloud-translate, google-generativeai, googleapis-common-protos, grpc-google-iam-v1, grpcio-status, orbax-checkpoint, proto-plus, t

In [3]:
!pip install tensorflow-metadata==1.15.0

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/pip/_vendor/pkg_resources/__init__.py", line 3070, in _dep_map
    return self.__dep_map
  File "/usr/local/lib/python3.10/dist-packages/pip/_vendor/pkg_resources/__init__.py", line 2863, in __getattr__
    raise AttributeError(attr)
AttributeError: _DistInfoDistribution__dep_map

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/commands/install.py", line 5

In [ ]:
import cv2
import math as m
from IPython.display import display, Video
from google.colab import files

# Install necessary libraries in Colab
if 'google.colab' in str(get_ipython()):
    !pip install opencv-python
    !pip install mediapipe

    # Upload your video file
    print("Please upload your video file")
    uploaded = files.upload()
    video_path = list(uploaded.keys())[0]  # Get the uploaded video file name
else:
    video_path = 'input.mp4'  # For local testing

import mediapipe as mp

# Initialize mediapipe pose and holistic classes
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()

# Helper functions
def findDistance(x1, y1, x2, y2):
    return m.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

def findAngle(x1, y1, x2, y2):
    try:
        if y1 == y2:  # Avoid division by zero if the points are vertically aligned
            return 90
        theta = m.acos((y2 - y1) * (-y1) / (m.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2) * y1))
        return int((180 / m.pi) * theta)
    except (ValueError, ZeroDivisionError):
        return 0  # Return 0 in case of any calculation errors

def sendWarning():
    print("Warning: Bad posture detected for more than 3 minutes!")

# Initialize counters
good_frames = 0
bad_frames = 0

# Font and color settings
font = cv2.FONT_HERSHEY_SIMPLEX
green = (127, 255, 0)
red = (50, 50, 255)
yellow = (0, 255, 255)
pink = (255, 0, 255)

# Video processing setup
cap = cv2.VideoCapture(video_path)

# Check if the video file is opened correctly
if not cap.isOpened():
    print(f"Error: Could not open video file {video_path}")
else:
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_size = (width, height)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    output_file = 'output.mp4'
    video_output = cv2.VideoWriter(output_file, fourcc, fps, frame_size)

    print("Processing video...")

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            print("End of video or null frame. Exiting...")
            break

        # Process the image with MediaPipe
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        keypoints = pose.process(image_rgb)

        # Check if pose landmarks are detected
        if keypoints and keypoints.pose_landmarks:
            lm = keypoints.pose_landmarks.landmark
            lmPose = mp_pose.PoseLandmark

            h, w = image.shape[:2]
            # Check if keypoints are valid to avoid index errors
            if (lmPose.LEFT_SHOULDER in lm and lmPose.RIGHT_SHOULDER in lm and
                lmPose.LEFT_EAR in lm and lmPose.LEFT_HIP in lm):
                # Get coordinates of landmarks
                l_shldr_x, l_shldr_y = int(lm[lmPose.LEFT_SHOULDER].x * w), int(lm[lmPose.LEFT_SHOULDER].y * h)
                r_shldr_x, r_shldr_y = int(lm[lmPose.RIGHT_SHOULDER].x * w), int(lm[lmPose.RIGHT_SHOULDER].y * h)
                l_ear_x, l_ear_y = int(lm[lmPose.LEFT_EAR].x * w), int(lm[lmPose.LEFT_EAR].y * h)
                l_hip_x, l_hip_y = int(lm[lmPose.LEFT_HIP].x * w), int(lm[lmPose.LEFT_HIP].y * h)

                # Calculate posture metrics
                offset = findDistance(l_shldr_x, l_shldr_y, r_shldr_x, r_shldr_y)
                neck_inclination = findAngle(l_shldr_x, l_shldr_y, l_ear_x, l_ear_y)
                torso_inclination = findAngle(l_hip_x, l_hip_y, l_shldr_x, l_shldr_y)

                # Posture feedback
                if neck_inclination < 40 and torso_inclination < 10:
                    bad_frames = 0
                    good_frames += 1
                    cv2.putText(image, f"Good Posture! {neck_inclination} | {torso_inclination}", (10, 30), font, 1, green, 2)
                else:
                    good_frames = 0
                    bad_frames += 1
                    cv2.putText(image, f"Bad Posture! {neck_inclination} | {torso_inclination}", (10, 30), font, 1, red, 2)

                # Time thresholds
                good_time = (1 / fps) * good_frames
                bad_time = (1 / fps) * bad_frames

                if bad_time > 180:  # 3 minutes threshold
                    sendWarning()

                # Draw landmarks for better visualization
                cv2.circle(image, (l_shldr_x, l_shldr_y), 7, yellow, -1)
                cv2.circle(image, (l_ear_x, l_ear_y), 7, pink, -1)
                cv2.circle(image, (l_hip_x, l_hip_y), 7, yellow, -1)
                cv2.line(image, (l_shldr_x, l_shldr_y), (l_ear_x, l_ear_y), green if good_frames else red, 4)

        video_output.write(image)

    cap.release()
    video_output.release()

    # Display the processed video output in the notebook
    print("Video processing complete! Displaying the result:")
    display(Video(output_file, embed=True))
